# 01 - Data Generation

Generate a realistic multi-modal restaurant dataset.
This notebook creates tabular data, text reviews, and downloads actual restaurant images.

In [21]:
import pandas as pd
import numpy as np
import os
import random
import urllib.request
import time

# Create directories
os.makedirs('dataset', exist_ok=True)
os.makedirs('restaurant_images', exist_ok=True)

print('Directories created')

Directories created


## Generate Realistic Restaurant Data

In [22]:
# Set random seed
np.random.seed(42)
random.seed(42)

# Restaurant data
cuisines = ['Italian', 'Asian', 'Mexican', 'American', 'Mediterranean', 'Indian', 'Chinese', 'French', 'Thai', 'Japanese']
neighborhoods = ['Downtown', 'Midtown', 'Uptown', 'Westside', 'Eastside', 'Northside', 'Southside']
prices = ['$', '$$', '$$$', '$$$$']

restaurant_names = [
    'La Bella Italia', 'Dragon Palace', 'El Mariachi', 'The American Grill',
    'Mediterranean Breeze', 'Taj Mahal', 'Golden Dragon', 'Le Petit Bistro',
    'Tokyo Sushi', 'Thai Orchid', 'Casa Mexicana', 'The Steakhouse',
    'Pasta Paradise', 'Spice Route', 'Ocean Blue', 'Mountain View',
    'City Lights', 'Garden Fresh', 'Sunset Cafe', 'Midnight Diner'
]

positive_reviews = [
    'Absolutely amazing! Best restaurant experience ever. Highly recommended!',
    'Excellent food and service. Will definitely come back.',
    'Fantastic atmosphere and delicious food. A must-visit!',
    'Outstanding quality. The chef really knows what they are doing.',
    'Perfect in every way. Exceeded all expectations.',
    'The flavors were incredible and the presentation was beautiful.',
    'Impeccable service from start to finish. Every dish was a masterpiece.',
    'A hidden gem! The portions were generous and the taste was authentic.',
    'Loved the ambiance and the staff were so welcoming. Food was superb.',
    'Best dining experience this year. The dessert was to die for!',
    'Everything was cooked to perfection. Highly recommended for special occasions.',
    'The freshness of ingredients really stood out. Will be coming back regularly.',
]

neutral_reviews = [
    'Good restaurant. Decent food and reasonable prices.',
    'Nice place. Food was okay, nothing special.',
    'Average experience. Could be better but not bad.',
    'Decent food. Service was a bit slow though.',
    'It is fine. Nothing to complain about, nothing to praise.',
    'The food was okay but the wait time was longer than expected.',
    'Reasonable prices for the quality. Not memorable but not bad either.',
    'The menu had good variety but the execution was inconsistent.',
    'A decent place for a quick meal. Nothing extraordinary.',
    'Service was friendly but the food took too long to arrive.',
    'The atmosphere was nice but the food was just average.',
    'Some dishes were good, others were disappointing. Mixed experience.',
]

negative_reviews = [
    'Disappointing. Would not recommend.',
    'Poor service and cold food. Avoid this place.',
    'Not worth the price. Very disappointing.',
    'Terrible experience. Will not return.',
    'Worst restaurant I have been to. Do not go there.',
    'The food was bland and overpriced. Service was rude.',
    'I had high expectations but left completely dissatisfied.',
    'Hygiene concerns were noticeable. Will not be coming back.',
    'Overcooked and under-seasoned. A complete waste of money.',
    'The portion sizes were tiny for the price charged. Very disappointed.',
    'Unpleasant smell and dirty tables. The staff seemed uninterested.',
    'One of the worst meals I have ever had. Stay away from this place.',
]

print('Data templates ready')

Data templates ready


In [23]:
# Generate 150 restaurants
restaurants = []

for i in range(150):
    rating = round(2.5 + np.random.beta(2, 1.5) * 2.5, 1)
    rating = min(5.0, max(2.5, rating))
    
    if rating >= 4.5:
        review_text = random.choice(positive_reviews)
        sentiment = 'positive'
    elif rating >= 4.0:
        review_text = random.choice(positive_reviews)
        sentiment = 'positive'
    elif rating >= 3.5:
        review_text = random.choice(neutral_reviews)
        sentiment = 'neutral'
    elif rating >= 3.0:
        review_text = random.choice(neutral_reviews)
        sentiment = 'neutral'
    else:
        review_text = random.choice(negative_reviews)
        sentiment = 'negative'
    
    review_count = int(50 + (rating - 2.5) * 100 + np.random.normal(0, 20))
    review_count = max(10, review_count)
    
    restaurants.append({
        'restaurant_id': i,
        'name': restaurant_names[i % len(restaurant_names)] if i < len(restaurant_names) else f'Restaurant {cuisines[i % len(cuisines)]} {i}',
        'rating': rating,
        'cuisine_type': cuisines[i % len(cuisines)],
        'price_range': prices[i % len(prices)],
        'review_count': review_count,
        'neighborhood': neighborhoods[i % len(neighborhoods)],
        'review_text': review_text,
        'sentiment': sentiment,
        'image_url': f'local_{i}',
        'is_high_tier': 1 if rating >= 4.0 else 0
    })

print(f'Generated {len(restaurants)} restaurants')

Generated 150 restaurants


In [24]:
# Create DataFrame
df = pd.DataFrame(restaurants)
print(f'Dataset shape: {df.shape}')
print(f'\nFirst few rows:')
print(df.head())

Dataset shape: (150, 11)

First few rows:
   restaurant_id                  name  rating   cuisine_type price_range  \
0              0       La Bella Italia     4.3        Italian           $   
1              1         Dragon Palace     3.7          Asian          $$   
2              2           El Mariachi     3.4        Mexican         $$$   
3              3    The American Grill     2.7       American        $$$$   
4              4  Mediterranean Breeze     4.5  Mediterranean           $   

   review_count neighborhood  \
0           225     Downtown   
1           190      Midtown   
2           144       Uptown   
3            39     Westside   
4           251     Eastside   

                                         review_text sentiment image_url  \
0  Absolutely amazing! Best restaurant experience...  positive   local_0   
1  Good restaurant. Decent food and reasonable pr...   neutral   local_1   
2   Average experience. Could be better but not bad.   neutral   local_2  

In [25]:
# Save tabular data
df.to_csv('dataset/restaurant_tabular_data.csv', index=False)
print('Saved tabular data')

Saved tabular data


In [26]:
# Save reviews
reviews_df = df[['restaurant_id', 'review_text', 'rating', 'sentiment']].copy()
reviews_df.to_csv('dataset/restaurant_reviews.csv', index=False)
print('Saved reviews')

Saved reviews


In [27]:
# Save image references
images_df = df[['restaurant_id', 'image_url']].copy()
images_df['local_path'] = images_df['restaurant_id'].apply(lambda x: f'restaurant_images/restaurant_{x}.jpg')
images_df.to_csv('dataset/restaurant_images.csv', index=False)
print('Saved image references')

Saved image references


## Download Actual Restaurant/Food Images

Download real restaurant and food images from multiple sources.
Each restaurant will get a different image by rotating through available sources.

In [28]:
# Free image sources for restaurant/food images
image_sources = [
    'https://images.unsplash.com/photo-1517248135467-4c7edcad34c4?w=800',  # Restaurant interior
    'https://images.unsplash.com/photo-1555396273-367ea4eb4db5?w=800',  # Restaurant food
    'https://images.unsplash.com/photo-1559339352-11d035aa65de?w=800',  # Restaurant
    'https://images.unsplash.com/photo-1552566626-52f8b828add9?w=800',  # Food
    'https://images.unsplash.com/photo-1504674900247-0877df9cc836?w=800',  # Food
    'https://images.unsplash.com/photo-1476224203421-9ac39bcb3327?w=800',  # Food
    'https://images.unsplash.com/photo-1540189549336-e6e99c3679fe?w=800',  # Food
    'https://images.unsplash.com/photo-1565299624946-b28f40a0ae38?w=800',  # Pizza
    'https://images.unsplash.com/photo-1568901346375-23c9450c58cd?w=800',  # Burger
    'https://images.unsplash.com/photo-1563245372-f21724e3856d?w=800',  # Dessert
    'https://images.unsplash.com/photo-1567620905732-2d1ec7ab7445?w=800',  # Food
    'https://images.unsplash.com/photo-1473093295043-cdd812d0e601?w=800',  # Food
    'https://images.unsplash.com/photo-1546069901-ba9599a7e63c?w=800',  # Food
    'https://images.unsplash.com/photo-1565958011703-44f9829ba187?w=800',  # Food
    'https://images.unsplash.com/photo-1482049016688-2d3e1b311543?w=800',  # Food
    'https://images.unsplash.com/photo-1504674900247-0877df9cc836?w=800',  # Food
    'https://images.unsplash.com/photo-1476718406336-bb5a4698b0c6?w=800',  # Food
    'https://images.unsplash.com/photo-1484723091739-30a097e8f929?w=800',  # Food
    'https://images.unsplash.com/photo-1499028344353-d617c2de8c6c?w=800',  # Food
    'https://images.unsplash.com/photo-1414235077428-338989a2e8c0?w=800',  # Restaurant interior
    'https://images.unsplash.com/photo-1514933651103-005eec06c04b?w=800',  # Restaurant
    'https://images.unsplash.com/photo-1533777857889-4be7c70b33f7?w=800',  # Restaurant interior
    'https://images.unsplash.com/photo-1550966871-3ed3cdb51f3a?w=800',  # Restaurant
    'https://images.unsplash.com/photo-1578474846511-04ba529f0b88?w=800',  # Fine dining
    'https://images.unsplash.com/photo-1559339352-11d035aa65de?w=800',  # Restaurant interior
    'https://images.unsplash.com/photo-1466978913421-dad2ebd01d17?w=800',  # Restaurant
    'https://images.unsplash.com/photo-1544148103-0773bf10d330?w=800',  # Restaurant interior
    'https://images.unsplash.com/photo-1550966871-3ed3cdb51f3a?w=800',  # Bar
    'https://images.unsplash.com/photo-1501339847302-ac426a4a7cbb?w=800',  # Coffee
    'https://images.unsplash.com/photo-1442512595331-e89e73853f31?w=800',  # Coffee shop
    'https://images.unsplash.com/photo-1554118811-1e0d58224f24?w=800',  # Cafe
    'https://images.unsplash.com/photo-1507048331197-7d4ac70811cf?w=800',  # Restaurant interior
    'https://images.unsplash.com/photo-1559339352-11d035aa65de?w=800',  # Food
    'https://images.unsplash.com/photo-1490645935967-10de6ba17061?w=800',  # Food plate
    'https://images.unsplash.com/photo-1432139555190-58524dae6a55?w=800',  # Steak
    'https://images.unsplash.com/photo-1540189549336-e6e99c3679fe?w=800',  # Salad
    'https://images.unsplash.com/photo-1563379926898-05f4575a45d8?w=800', # Sushi
    'https://images.unsplash.com/photo-1559847844-5315695dadae?w=800',  # Pasta
    'https://images.unsplash.com/photo-1551218808-94e220e084d2?w=800',  # Restaurant
]

print(f'Image sources: {len(image_sources)}')

Image sources: 19


In [29]:
def download_image(url, filepath, timeout=15):
    """
    Download an image from URL to filepath
    """
    try:
        if url and url.startswith('http'):
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=timeout) as response:
                with open(filepath, 'wb') as f:
                    f.write(response.read())
            return True
    except Exception as e:
        return False

In [30]:
# Download images for all restaurants - rotate through different sources
downloaded = 0
failed = 0

for idx, row in df.iterrows():
    filepath = f'restaurant_images/restaurant_{row["restaurant_id"]}.jpg'
    
    # Rotate through different image sources for each restaurant
    url_idx = idx % len(image_sources)
    url = image_sources[url_idx]
    
    if download_image(url, filepath):
        downloaded += 1
    else:
        failed += 1
    
    if (idx + 1) % 30 == 0:
        print(f'Downloaded {downloaded}, Failed {failed}, Total {idx + 1}')
    
    time.sleep(0.3)  # Rate limiting

print(f'\nDownload complete: {downloaded} downloaded, {failed} failed')

Downloaded 28, Failed 2, Total 30
Downloaded 54, Failed 6, Total 60
Downloaded 82, Failed 8, Total 90
Downloaded 108, Failed 12, Total 120
Downloaded 135, Failed 15, Total 150

Download complete: 135 downloaded, 15 failed


In [31]:
# Verify images
image_files = [f for f in os.listdir('restaurant_images') if f.endswith('.jpg')]
print(f'Images in folder: {len(image_files)}')

Images in folder: 135


In [32]:
# If some failed, copy from successful downloads
if failed > 0:
    print(f'\n{failed} images failed. Copying from successful downloads.')
    
    from PIL import Image
    
    # Get list of successful images
    successful_images = [f for f in os.listdir('restaurant_images') if f.endswith('.jpg')]
    
    for idx, row in df.iterrows():
        filepath = f'restaurant_images/restaurant_{row["restaurant_id"]}.jpg'
        if not os.path.exists(filepath):
            # Copy a random successful image
            if successful_images:
                template_path = f'restaurant_images/{random.choice(successful_images)}'
                template_img = Image.open(template_path)
                template_img.save(filepath)
                downloaded += 1
                failed -= 1
    
    print(f'After copying: {downloaded} total images')


15 images failed. Copying from successful downloads.
After copying: 150 total images


In [33]:
# Final check
image_files = [f for f in os.listdir('restaurant_images') if f.endswith('.jpg')]
print(f'Final image count: {len(image_files)}')

Final image count: 150


## Summary

Data generation complete:
- **150 restaurants** with realistic tabular data
- **Reviews** for sentiment analysis
- **Images** downloaded from actual food/restaurant image sources (different images per restaurant)
- **Multi-modal dataset** ready for the 5 models

All data is saved and ready for preprocessing.